# UNLOCKD Risk Simulation (Monte Carlo)

This notebook runs GBM-based simulations to estimate conservative PV (5th percentile)
for locked tokens. Use it to calibrate governance parameters and stress test
asset classes.

In [ ]:
import numpy as np
import pandas as pd

P0 = 10.0
Q = 1000.0
r = 0.05
vols = [0.30, 0.50, 0.70]
months = list(range(0, 37, 3))
paths = 10_000
seed = 42

rng = np.random.default_rng(seed)

In [ ]:
def simulate_pv(P0, Q, r, sigma, month, n_paths, rng):
    t = month / 12.0
    if t == 0:
        pv = P0 * Q
        return pv, pv

    z = rng.standard_normal(n_paths)
    drift = (r - 0.5 * sigma ** 2) * t
    diffusion = sigma * np.sqrt(t) * z
    pt = P0 * np.exp(drift + diffusion)
    pv = np.exp(-r * t) * Q * pt
    return pv.mean(), np.percentile(pv, 5)

In [ ]:
rows = []
for m in months:
    row = {"Months": m}
    for sigma in vols:
        mean_pv, p5_pv = simulate_pv(P0, Q, r, sigma, m, paths, rng)
        row[f"Vol {int(sigma*100)}% Mean PV"] = round(mean_pv, 0)
        row[f"Vol {int(sigma*100)}% 5th % PV"] = round(p5_pv, 0)
    rows.append(row)

result = pd.DataFrame(rows)
result

In [ ]:
# LTV curve + borrow limits derived from 5th percentile PV

def ltv_for_months(month):
    return max(0.25, 0.40 - (0.15 / 36.0) * month)

ltv = result["Months"].apply(ltv_for_months)

borrow_rows = pd.DataFrame({
    "Months": result["Months"],
    "LTV %": (ltv * 100).round(2),
    "Borrow 30%": (result["Vol 30% 5th % PV"] * ltv).round(0),
    "Borrow 50%": (result["Vol 50% 5th % PV"] * ltv).round(0),
    "Borrow 70%": (result["Vol 70% 5th % PV"] * ltv).round(0),
})

borrow_rows

In [ ]:
# Plot borrow limits (5th percentile PV × LTV)
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(borrow_rows["Months"], borrow_rows["Borrow 30%"], label="Borrow 30%")
ax.plot(borrow_rows["Months"], borrow_rows["Borrow 50%"], label="Borrow 50%")
ax.plot(borrow_rows["Months"], borrow_rows["Borrow 70%"], label="Borrow 70%")

ax.set_title("Borrow Limits (Conservative)")
ax.set_xlabel("Months to unlock")
ax.set_ylabel("Borrow limit")
ax.legend()
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
# Export results to CSV for governance reviews
output_path = "risk_sim_results.csv"
result.to_csv(output_path, index=False)
output_path

In [ ]:
# Plot 5th percentile PV by months for each volatility band
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
for sigma in vols:
    column = f"Vol {int(sigma * 100)}% 5th % PV"
    ax.plot(result["Months"], result[column], label=column)

ax.set_title("Conservative PV (5th percentile)")
ax.set_xlabel("Months to unlock")
ax.set_ylabel("PV")
ax.legend()
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()